# 주택담보대출 상환금 계산기

원리금균등, 원금균등, 만기일시 상환 방식을 비교합니다. 아래 설정값을 변경한 뒤 셀을 순서대로 실행하세요.

In [1]:
# 대출 설정값
PRINCIPAL = 100_000_000  # 대출 금액(원)
YEARS = 30               # 대출 기간(년)
ANNUAL_RATE = 4.5        # 연이율(%)

In [2]:
from typing import Any, Dict
import pandas as pd
from IPython.display import display

class MortgageCalculator:
    def __init__(self, principal: float, years: int, annual_rate: float):
        if principal <= 0: raise ValueError('대출금액은 0보다 커야 합니다.')
        if years <= 0: raise ValueError('대출기간은 1년 이상이어야 합니다.')
        if annual_rate < 0: raise ValueError('연이율은 0% 이상이어야 합니다.')
        self.principal = float(principal)
        self.years = int(years)
        self.months = self.years * 12
        self.annual_rate = float(annual_rate)
        self.monthly_rate = (self.annual_rate / 100) / 12

    def calculate_equal_principal_and_interest(self) -> Dict[str, Any]:
        r, n, p = self.monthly_rate, self.months, self.principal
        payment = p / n if r == 0 else p * (r * (1 + r) ** n) / ((1 + r) ** n - 1)
        schedule, balance, total_interest = [], p, 0.0
        for month in range(1, n + 1):
            interest = balance * r
            principal_payment = balance if month == n else payment - interest
            actual_payment = principal_payment + interest if month == n else payment
            balance -= principal_payment
            if abs(balance) < 1e-5: balance = 0.0
            total_interest += interest
            schedule.append({'month': month, 'monthly_payment': round(actual_payment), 'principal_payment': round(principal_payment), 'interest_payment': round(interest), 'remaining_balance': round(balance)})
        return self._result('원리금균등상환', schedule, total_interest)

    def calculate_equal_principal(self) -> Dict[str, Any]:
        r, n, p = self.monthly_rate, self.months, self.principal
        monthly_principal = p / n
        schedule, balance, total_interest = [], p, 0.0
        for month in range(1, n + 1):
            interest = balance * r
            payment = monthly_principal + interest
            balance -= monthly_principal
            if abs(balance) < 1e-5: balance = 0.0
            total_interest += interest
            schedule.append({'month': month, 'monthly_payment': round(payment), 'principal_payment': round(monthly_principal), 'interest_payment': round(interest), 'remaining_balance': round(balance)})
        return self._result('원금균등상환', schedule, total_interest)

    def calculate_bullet_maturity(self) -> Dict[str, Any]:
        r, n, p = self.monthly_rate, self.months, self.principal
        interest = p * r
        schedule = []
        for month in range(1, n + 1):
            last = month == n
            schedule.append({'month': month, 'monthly_payment': round(interest + (p if last else 0)), 'principal_payment': round(p if last else 0), 'interest_payment': round(interest), 'remaining_balance': round(0 if last else p)})
        return self._result('만기일시상환', schedule, interest * n)

    def _result(self, name, schedule, total_interest):
        return {'method_name': name, 'first_month_payment': schedule[0]['monthly_payment'], 'last_month_payment': schedule[-1]['monthly_payment'], 'total_interest': round(total_interest), 'total_payment': round(self.principal + total_interest), 'schedule': schedule}

def calculate_all(principal, years, annual_rate):
    calculator = MortgageCalculator(principal, years, annual_rate)
    return [calculator.calculate_equal_principal_and_interest(), calculator.calculate_equal_principal(), calculator.calculate_bullet_maturity()]

In [3]:
# 세 가지 상환 방식 요약 비교
results = calculate_all(PRINCIPAL, YEARS, ANNUAL_RATE)
summary = pd.DataFrame([{
    '상환 방식': r['method_name'],
    '첫 달 납입액': r['first_month_payment'],
    '마지막 달 납입액': r['last_month_payment'],
    '총 납부 이자': r['total_interest'],
    '총 상환 금액': r['total_payment'],
} for r in results])
summary_display = summary.copy()
for column in summary_display.columns[1:]:
    summary_display[column] = summary_display[column].map(lambda value: f'{value:,.0f}원')
display(summary_display)

,상환 방식,첫 달 납입액,마지막 달 납입액,총 납부 이자,총 상환 금액
0,원리금균등상환,"506,685원","506,685원","82,406,712원","182,406,712원"
1,원금균등상환,"652,778원","278,819원","67,687,500원","167,687,500원"
2,만기일시상환,"375,000원","100,375,000원","135,000,000원","235,000,000원"


In [4]:
# 원하는 상환 방식의 월별 일정 확인 (0: 원리금균등, 1: 원금균등, 2: 만기일시)
SELECTED_METHOD = 0
schedule = pd.DataFrame(results[SELECTED_METHOD]['schedule']).rename(columns={
    'month': '회차', 'monthly_payment': '월 납입액', 'principal_payment': '원금',
    'interest_payment': '이자', 'remaining_balance': '남은 원금'
})
schedule_display = schedule.copy()
for column in schedule_display.columns[1:]:
    schedule_display[column] = schedule_display[column].map(lambda value: f'{value:,.0f}원')
display(schedule_display)

,회차,월 납입액,원금,이자,남은 원금
0,1,"506,685원","131,685원","375,000원","99,868,315원"
1,2,"506,685원","132,179원","374,506원","99,736,136원"
2,3,"506,685원","132,675원","374,011원","99,603,461원"
3,4,"506,685원","133,172원","373,513원","99,470,288원"
4,5,"506,685원","133,672원","373,014원","99,336,617원"
...,...,...,...,...,...
355,356,"506,685원","497,291원","9,394원","2,007,882원"
356,357,"506,685원","499,156원","7,530원","1,508,726원"
357,358,"506,685원","501,028원","5,658원","1,007,699원"
358,359,"506,685원","502,906원","3,779원","504,792원"
